# Week 3 — VADER sentiment scoring

**Input:** `data/processed/reviews_clean.csv`

**Output:** `data/processed/reviews_sentiment.csv` (per-review scores), `data/processed/sentiment_summary.csv` (per-entity/platform rollup, feeds the Week 4 proxy-NPS calculation)

In [1]:
from pathlib import Path

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

analyzer = SentimentIntensityAnalyzer()


def score(text):
    s = analyzer.polarity_scores(text)
    return s["compound"], s["pos"], s["neu"], s["neg"]


def label(compound):
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

In [2]:
df = pd.read_csv(PROCESSED / "reviews_clean.csv")

scores = df["text"].apply(score)
df[["compound", "pos", "neu", "neg"]] = pd.DataFrame(scores.tolist(), index=df.index)
df["sentiment"] = df["compound"].apply(label)

df.to_csv(PROCESSED / "reviews_sentiment.csv", index=False)
print(f"scored {len(df)} reviews -> data/processed/reviews_sentiment.csv")
df.head()

scored 1302 reviews -> data/processed/reviews_sentiment.csv


,entity,platform,review_id,rating,text,date,app_version,title,compound,pos,neu,neg,sentiment
0,hdfc_bank,app_store,14285276560,5,Great,NaN,NaN,Rating,0.6249,1.000,0.000,0.000,positive
1,hdfc_bank,app_store,14285125471,5,It’s better app,NaN,NaN,Good one,0.4404,0.592,0.408,0.000,positive
2,hdfc_bank,app_store,14284893772,1,Very poor sarvice i think i currant account i ...,NaN,NaN,Sarvice,-0.5256,0.000,0.834,0.166,negative
3,hdfc_bank,app_store,14284815366,5,Easy handling,NaN,NaN,Good App,0.4404,0.744,0.256,0.000,positive
4,hdfc_bank,app_store,14284475135,1,new app very complicated,NaN,NaN,aa,0.0000,0.000,1.000,0.000,neutral


## Per-entity / platform rollup

In [3]:
summary = (
    df.groupby(["entity", "platform"])
    .agg(
        n_reviews=("text", "count"),
        avg_star_rating=("rating", "mean"),
        avg_compound=("compound", "mean"),
        pct_positive=("sentiment", lambda s: (s == "positive").mean() * 100),
        pct_negative=("sentiment", lambda s: (s == "negative").mean() * 100),
    )
    .round(3)
    .reset_index()
)
summary.to_csv(PROCESSED / "sentiment_summary.csv", index=False)
print("-> data/processed/sentiment_summary.csv")
summary

-> data/processed/sentiment_summary.csv


,entity,platform,n_reviews,avg_star_rating,avg_compound,pct_positive,pct_negative
0,hdfc_bank,app_store,448,3.179,0.173,53.795,23.884
1,hdfc_bank,google_play,177,2.113,-0.153,27.119,50.847
2,jupiter,app_store,495,1.717,-0.027,40.202,47.677
3,jupiter,google_play,182,2.571,0.062,49.451,33.516
